# Phase 3 status and next steps

**Question.** What is done, what is only preliminary, and what is deferred to HPC or to a decision?

Sources: `docs/phase3-findings.md` and `docs/hpc-runbook.md`. The tables below are read from those files. Edit the docs, not this notebook.
Everything quantitative in Phase 3 is **synthetic**.

## What this shows / does not show

**Shows** the status of each claim and each open item at handoff.
**Does not show** new results. For evidence read notebooks 07 to 09 and the finding IDs in the Research Panel (`.research/bin/research current`).

In [1]:
from pathlib import Path
import re
import pandas as pd
pd.set_option("display.max_colwidth", 140); pd.set_option("display.width", 170)
ROOT = Path("..").resolve()
doc = (ROOT / "docs" / "phase3-findings.md").read_text()

def section(title):
    m = re.search(rf"^## {re.escape(title)}\n(.*?)(?=^## |\Z)", doc, re.S | re.M)
    return m.group(1)

rows = [[c.strip() for c in l.strip().strip("|").split("|")] for l in section("Supported by evidence so far").splitlines()
        if l.startswith("|") and not l.startswith("|---")]
done = pd.DataFrame(rows[1:], columns=rows[0])
done

,Claim,Kind,Finding,Strength
0,The geodesign pipeline runs on an unlike synthetic region with config + asset changes; only the raw-data loader needed a role-name change,M/S,"T-023, `docs/transfer.md`",tested on one synthetic region; no real second municipality
1,Population layer conserves totals under regrid; metrics match hand computation,M,`tests/test_population.py`,unit tests
2,"`vanleer` scheme: closure 3e-13, positive, order ~1.8-2.4 periodic test, nonlinear, needs sub-stepping, legacy outputs bit-identical (21...",M/N,`results/phase3-scheme/`,one test family
3,"At dt 25 s the 500 m exponential field has ~5-6% (lower bound) space-time error vs 125 m, observed order ~1.0-1.2; mesh error dominates ...",N,F-025,"one case, one wind"
4,Unknown wind direction error raises field error ~1.4-1.6x at 5 deg and 3.5-3.8x at 30 deg (fixed wrong wind); for >=10 deg it outweighs ...,S,F-029,"toy domain, 32 test cases, inverse crime"
5,A wind-ensemble estimator recovers part of the loss only when the true wind is near an ensemble member; its intervals are overconfident ...,S,F-029,same
6,"Outer-domain half-width needed for <1% exposure ~ U T /2 for the reversal scenarios (40 km here), set by the return window, grows with T",S,F-031,one source geometry


## Preliminary or contested

In [2]:
def bullets(title):
    return [re.sub(r"\s+", " ", b).strip("* ").strip() for b in re.split(r"\n(?=\* )", section(title).strip()) if b.strip().startswith("*")]
pd.DataFrame({"item": [b.split(".")[0][:90] for b in bullets("Preliminary or contested")], "summary": bullets("Preliminary or contested")})

,item,summary
0,"Population-aware design (F-026, F-030)","Population-aware design (F-026, F-030).** With greedy selection no objective effect is established (EXP-017). With exact selection (EXP-..."
1,Greedy selection is far from optimal at 2-4 sensors on the design objective (F-030)**: med,Greedy selection is far from optimal at 2-4 sensors on the design objective (F-030)**: median excess 6.7x / 8.9x / 3.0x at budgets 2/3/4...
2,Prescribed-wind scenarios (F-031),Prescribed-wind scenarios (F-031).** Channel offset (-43% exposure) is a placement effect; the vortex is -10%; reversals raise exposure ...


## Deferred (needs HPC or a decision)

In [3]:
pd.DataFrame({"item": [b[:70] for b in bullets("Not done (needs HPC or a decision)")], "detail": bullets("Not done (needs HPC or a decision)")})

,item,detail
0,"EXP-015 stage 2: do sensor-design decisions (displacement, regret) con","EXP-015 stage 2: do sensor-design decisions (displacement, regret) converge with mesh/dt relative to a resampling baseline? Code built, ..."
1,EXP-016: Belauri-scale population-aware design (specified in `docs/hpc,"EXP-016: Belauri-scale population-aware design (specified in `docs/hpc-runbook.md`, not implemented)."
2,"S5a: wind-robust and population-robust placement (conditional, not sta","S5a: wind-robust and population-robust placement (conditional, not started)."
3,Real population data (WorldPop/GHS-POP) and real meteorology (ERA5): f,"Real population data (WorldPop/GHS-POP) and real meteorology (ERA5): feasibility notes only (`docs/population-data.md`, `docs/winds.md`)..."
4,"Review-requested controls: wind-error toy with an offset prior grid, a","Review-requested controls: wind-error toy with an offset prior grid, alpha sensitivity for the known-wind baseline, model error at zero ..."


## HPC runbook: state of the code at handoff

In [4]:
rb = (ROOT / "docs" / "hpc-runbook.md").read_text()
blk = rb.split("## 10. State of the code at handoff")[1].split("## 11.")[0]
rows = [[c.strip() for c in l.strip().strip("|").split("|")] for l in blk.splitlines() if l.startswith("|") and not l.startswith("|---")]
pd.DataFrame(rows[1:], columns=rows[0])

,Item,Code (commit),Tests,Run?,Notes
0,"Solver interface, FiPy routing, regression gate","`forward.py`, `inverse.py` (Phase 2)",yes,gate captured (21 files),rerun the gate on the cluster
1,"Population layer (counts, conservative regrid, fixtures, weights, metrics)",`src/pravah/population.py` (fe9e1da),`tests/test_population.py`,n/a,no real raster used yet
2,Resolution benchmark (T-025),`scripts/bench_resolution.py` (96cb93f),smoke only,"yes, `results/phase3-bench/resolution_cost.csv`",125 m level is throw-away `geo125/`
3,vanleer convection (T-026),`forward.py` (706d1d2),`tests/test_scheme_vanleer.py`,`results/phase3-scheme/` (42ee6ab),"nonlinear, not usable with cached responses"
4,EXP-015 stage 1,`scripts/convergence_stage1.py` (fd6c19a),`tests/test_convergence_stage1.py`,"yes RUN-0028 (228163e), F-022 superseded by F-025",needs `geo125/` to rerun
5,EXP-015 stage 2,"`scripts/convergence_stage2.py` (2eb96e2), config",`tests/test_convergence_stage2.py` (38872b0),**no**,section 5; about 2 h
6,EXP-017 toy population design,`scripts/population_design.py` (b6d93b5),`tests/test_population_design.py`,"yes RUN-0030 (3f8c8b2), F-023 superseded by F-026","alpha 0.1, 16 test cases"
7,EXP-018 prescribed winds,"`src/pravah/winds.py` (887abbd), `scripts/prescribed_winds.py` (c000620), `docs/winds.md` (5644935)",`tests/test_winds.py`,"yes RUN-0029 (3f8c8b2), F-024 superseded by F-027, F-031",ridge amplitude too weak; finer grid is an HPC follow-up
8,EXP-019 toy wind error,"`src/pravah/windunc.py` (edc2cb9), `scripts/wind_error.py` (885d88b)",`tests/test_windunc.py`,pilot committed (d26be8b); production RUN-0031 uncommitted when written,verify before relying
9,EXP-016 Belauri population design,`scripts/population_belauri.py` (61f8ae4),`tests/test_population_belauri.py`,**no** (smoke + pilot only),section 6; in the one-job config


## Ranked recommendations for Phase 4

In [5]:
recs = re.findall(r"^\d\. (.*)$", section("Phase 4 recommendations (ranked)"), re.M)
pd.DataFrame({"rank": range(1, len(recs) + 1), "recommendation": recs})

,rank,recommendation
0,1,Run EXP-015 stage 2 on HPC (plus the optional reference-mesh replicate baseline and a 125 m design stage); decide the mesh for design.
1,2,Re-evaluate Phase 2 small-budget layouts with exact or swap-search selection and score greedy on test; rerun geodesign/geodesign_robust ...
2,3,"Build and run the Belauri-scale EXP-016 with exact/swap selection, replicated population draws, design-split control, bootstrap over pla..."
3,4,Belauri-scale wind-error study with the review controls; then S5a (robust placement) if wind error is still design-relevant at scale.
4,5,One real dataset each: WorldPop/GHS-POP population for Belauri; a bounded ERA5 extract entering only through `winds.segments_from_series...


## Added after the handoff document

The two docs above were written at handoff. Since then these items changed. The rows below are written by hand from the repository state. Check `.research/bin/research current` for finding status.

In [6]:
pd.DataFrame([
    ["EXP-022 greedy vs best-found on test (toy)", "done", "F-032", "greedy test area RMSE is 1.08 to 6.8 times the best-found one; only 4 source placements"],
    ["EXP-023 wind-error controls (toy)", "done, finding reviewed (status preliminary)", "F-034 (supersedes F-033)", "alpha, offset prior, model error K x1.2, base-wind rotation; inverse crime, reduced grids, 10 random layouts"],
    ["HPC package", "ready, not run", "docs/hpc-runbook.md, scripts/hpc_run.py, slurm/phase3.sbatch, slurm/setup_env.sh", "one job, phases in sequence, CPU only; placeholders to fill"],
    ["EXP-016 Belauri-scale population design", "code ready, not run", "scripts/population_belauri.py", "exact enumeration at budgets 2-4, swap search at 5 and 8; the table in runbook section 10 still says not written"],
    ["EXP-015 stage 2 (mesh decision)", "code ready, not run", "scripts/convergence_stage2.py", "all Belauri-scale sensor conclusions stay conditional on it"],
], columns=["item", "status", "where", "note"])

,item,status,where,note
0,EXP-022 greedy vs best-found on test (toy),done,F-032,greedy test area RMSE is 1.08 to 6.8 times the best-found one; only 4 source placements
1,EXP-023 wind-error controls (toy),"done, finding reviewed (status preliminary)",F-034 (supersedes F-033),"alpha, offset prior, model error K x1.2, base-wind rotation; inverse crime, reduced grids, 10 random layouts"
2,HPC package,"ready, not run","docs/hpc-runbook.md, scripts/hpc_run.py, slurm/phase3.sbatch, slurm/setup_env.sh","one job, phases in sequence, CPU only; placeholders to fill"
3,EXP-016 Belauri-scale population design,"code ready, not run",scripts/population_belauri.py,"exact enumeration at budgets 2-4, swap search at 5 and 8; the table in runbook section 10 still says not written"
4,EXP-015 stage 2 (mesh decision),"code ready, not run",scripts/convergence_stage2.py,all Belauri-scale sensor conclusions stay conditional on it


## Real inputs now available

Both inputs are in `data/`. **Neither is used in any experiment yet.**

| Input | File | What it holds |
|---|---|---|
| Population | `data/population/belauri_worldpop_2025_CN_{250m,500m}.npy` + `belauri_worldpop_2025_CN.json` | WorldPop R2025A constrained counts (people per cell), Nepal 2025, 100 m source, regridded to 250 m and 500 m with counts conserved |
| Wind | `data/era5/belauri_era5_10m_wind.nc` + `.json` | ERA5 hourly 10 m wind, u and v in m/s, 0.25 degree grid |

In [7]:
import json
import numpy as np
pop = np.load(ROOT / "data" / "population" / "belauri_worldpop_2025_CN_500m.npy")
meta = json.load(open(ROOT / "data" / "population" / "belauri_worldpop_2025_CN.json"))
print("population 500 m grid:", pop.shape, "cells; total in grid:", round(float(pop.sum())), "people;", meta["source"], "|", meta["license"])
try:
    import xarray as xr
    w = xr.open_dataset(ROOT / "data" / "era5" / "belauri_era5_10m_wind.nc")
    print("ERA5:", dict(w.sizes), "variables", list(w.data_vars))
except Exception as e:
    print("ERA5 file not read here:", type(e).__name__)

population 500 m grid: (65, 66) cells; total in grid: 184030 people; WorldPop Global 2 (R2025A), Nepal, 2025, constrained, 100 m | CC BY 4.0 (WorldPop; verify on the dataset page before redistribution)


ERA5: {'valid_time': 1488, 'latitude': 9, 'longitude': 9} variables ['u10', 'v10']


Caveats. WorldPop is a modelled estimate with a licence to check before redistribution. ERA5 at 0.25 degrees cannot resolve mountain-induced flow near Belauri.
A bounded ERA5 extract may enter only through `winds.segments_from_series`, with its coarseness stated.

## Limitations

* The tables mirror the two docs at the time this notebook was run. If the docs change, rerun the notebook.
* Finding status in Research OS may be newer than the docs. Check `.research/bin/research current`.
* All claims are synthetic and from a research testbed. None is a validated forecast and none is a deployment recommendation.